# DeepOptimizedNN Training Pipeline

This notebook implements and explains the **DeepOptimizedNN** — a deep feedforward neural network for classification tasks.  
It includes model definition, training/evaluation pipeline, and in-depth commentary on each design choice.


In [3]:
# 1. Imports and Setup

import uuid
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import os
from machine_learning.models.utils import bar_plot
 

In [5]:
# 1. The Model: DeepOptimizedNN

class DeepOptimizedNN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DeepOptimizedNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, output_dim)
        )

    def forward(self, x):
        return self.net(x)


### Pregled Arhitekture

Ovo je **duboka neuronska mreža (Deep Neural Network, DNN)** napravljena pomoću PyTorch-ovog `nn.Sequential`, koji omogućava linearno slaganje slojeva.

| Sloj | Tip | Svrha |
|------|------|--------|
| `nn.Linear(input_dim, 128)` | Gusto povezani sloj | Projektuje ulazne karakteristike u 128-dimenzionalni latentni prostor |
| `nn.BatchNorm1d(128)` | Normalizacija po serijama | Stabilizuje i ubrzava treniranje |
| `nn.LeakyReLU()` | Aktivacija | Omogućava prolaz gradijenta i za negativne ulaze |
| `nn.Dropout(0.3)` | Regularizacija | Nasumično poništava 30% aktivacija da bi sprečio preučenost |
| `nn.Linear(128, 64)` + BatchNorm/ReLU/Dropout | Skriveni sloj | Uči složenije obrasce |
| `nn.Linear(64, 32)` + BatchNorm/ReLU | Kompresija | Izvlači apstraktnije osobine |
| `nn.Linear(32, 16)` + ReLU | Pretposlednji sloj | Smanjuje dimenzionalnost reprezentacije |
| `nn.Linear(16, output_dim)` | Izlazni sloj | Generiše logite za klasifikaciju |

**Dizajnerski razlozi:**
- Postepeno smanjivanje dimenzija slojeva = hijerarhijsko izvlačenje osobina  
- Kombinacija LeakyReLU i ReLU = otpornost na „mrtve neurone“  
- BatchNorm + Dropout = brža konvergencija + bolja generalizacija  
- Rezultat: univerzalni duboki klasifikator

### Pregled Arhitekture

Ovo je **duboka neuronska mreža (Deep Neural Network, DNN)** napravljena pomoću PyTorch-ovog `nn.Sequential`, koji omogućava linearno slaganje slojeva.

| Sloj | Tip | Svrha |
|------|------|--------|
| `nn.Linear(input_dim, 128)` | Gusto povezani sloj | Projektuje ulazne karakteristike u 128-dimenzionalni latentni prostor |
| `nn.BatchNorm1d(128)` | Normalizacija po serijama | Stabilizuje i ubrzava treniranje |
| `nn.LeakyReLU()` | Aktivacija | Omogućava prolaz gradijenta i za negativne ulaze |
| `nn.Dropout(0.3)` | Regularizacija | Nasumično poništava 30% aktivacija da bi sprečio preučenost |
| `nn.Linear(128, 64)` + BatchNorm/ReLU/Dropout | Skriveni sloj | Uči složenije obrasce |
| `nn.Linear(64, 32)` + BatchNorm/ReLU | Kompresija | Izvlači apstraktnije osobine |
| `nn.Linear(32, 16)` + ReLU | Pretposlednji sloj | Smanjuje dimenzionalnost reprezentacije |
| `nn.Linear(16, output_dim)` | Izlazni sloj | Generiše logite za klasifikaciju |

**Dizajnerski razlozi:**
- Postepeno smanjivanje dimenzija slojeva = hijerarhijsko izvlačenje osobina  
- Kombinacija LeakyReLU i ReLU = otpornost na „mrtve neurone“  
- BatchNorm + Dropout = brža konvergencija + bolja generalizacija  
- Rezultat: univerzalni duboki klasifikator


---

## 4. Objašnjenja Hiperparametara i Razlozi za Izbor

### 1. `epochs = 40`
- Maksimalan broj prolazaka kroz ceo skup podataka.  
- Rano zaustavljanje prekida treniranje ako nema napretka.  
- Prenizak broj → nedovoljno naučeno; previsok broj → preučenost.

### 2. `batch_size = 256`
- Broj uzoraka po jednoj optimizacionoj iteraciji.  
- 256 omogućava stabilne gradijente i pouzdanu statistiku u BatchNorm slojevima.  

### 3. `lr = 1e-3`
- Veličina koraka tokom ažuriranja težina.  
- Standardna vrednost za AdamW, balans između brzine konvergencije i stabilnosti.  

### 4. `patience = 5`
- Prekida treniranje ako se validacioni gubitak ne poboljša tokom 5 epoha.  
- Sprečava preučenost i nepotrebno trošenje resursa.  

### 5. `max_train_samples = 60000`
- Ograničava količinu podataka radi bržeg treniranja i testiranja.  
- Nasumično uzorkuje podskup ako je dataset ogroman.  

### 6. `Dropout` slojevi = 0.3, 0.25
- Nasumično „isključuju“ neurone da bi sprečili međusobnu zavisnost (ko-adaptaciju).  
- Veći dropout u većim slojevima.  

### 7. `Weight Decay = 1e-4`
- L2 regularizacija koja smanjuje veličinu težina i sprečava prenaučenost.  
- Pomaže generalizaciji modela.  

### 8. Optimizator = `AdamW`
- Adaptivni gradijentni metod sa pravilnim razdvajanjem težinske regularizacije.  
- Brz, stabilan i dobro generalizuje.  

### 9. `BatchNorm1d`
- Normalizuje aktivacije unutar mini-serije.  
- Stabilizuje treniranje i omogućava veće stope učenja.  

### 10. Aktivacione funkcije
- **LeakyReLU** u prvom sloju (sprečava mrtve neurone).  
- **ReLU** u kasnijim slojevima (efikasna nakon normalizacije).  

### 11. `GradScaler` i `autocast`
- Treniranje sa mešanom preciznošću (FP16/FP32).  
- Smanjuje upotrebu memorije i ubrzava treniranje na GPU uređajima.  

### 12. Rano zaustavljanje + `best_state`
- Čuva model sa najboljim validacionim gubitkom.  
- Sprečava korišćenje preučenih težina.  

### 13. Vizuelizacija krive učenja
- Prikazuje krivu treniranja i validacije radi praćenja konvergencije i preučenosti.  

### 14. `classification_report` i bar grafikon
- `classification_report` prikazuje preciznost, odziv i F1 meru po klasama.  
- Bar grafikon vizuelno prikazuje performanse po klasama.  


## 15. Pregled Interakcija i Kompromisa

| Hiperparametar | Povećanje vodi ka → | Smanjenje vodi ka → |
|----------------|----------------------|----------------------|
| **epochs** | Bolja konvergencija, moguća preučenost | Brže treniranje, moguća nedoučenost |
| **batch_size** | Stabilniji gradijenti, manja generalizacija | Veća varijabilnost gradijenata, potencijalno bolja generalizacija |
| **lr (learning rate)** | Brža konvergencija, rizik od nestabilnosti | Stabilnije učenje, sporija konvergencija |
| **dropout** | Manja preučenost, sporije učenje | Brže učenje, veći rizik od preučenosti |
| **patience** | Veća tolerancija na oscilacije u gubitku | Može prerano zaustaviti treniranje |
| **weight_decay** | Glatkiji model, manja preučenost | Veća fleksibilnost, veći rizik od preučenosti |
